# 04 — Silver: For-Hire Vehicle (FHV)
Reads FHV records from Bronze (tlc_bronze.fhv_raw), applies data quality rules, enriches with zone metadata via broadcast join, builds the normalized Silver schema, and writes valid records to tlc_silver.trips_fhv.

In [ ]:
from src.core.spark import get_spark
from src.core.mongo import read_mongo, write_mongo
from src.silver.rules import FHV_RULES, apply_rules, print_rejection_summary
from src.silver.enrichment import load_zone_lookup, enrich_trip_locations
from src.silver.silver import build_fhv_silver
from src.core.audit import ControlManager, ExecutionStatus, get_audit_db
from src.core.config import settings
from pyspark.sql import functions as F

In [ ]:
YEARS_TO_PROCESS = [2024, 2025]

In [ ]:
spark = get_spark(app_name="TLC_Silver_FHV")

In [ ]:
control = ControlManager("silver_fhv")
execution = control.start({"years": YEARS_TO_PROCESS, "vehicle_type": "fhv"})
run_id = execution.execution_id

In [ ]:
df_bronze = read_mongo(spark, settings.MONGO_DB_BRONZE, "fhv_raw")
if "_meta" in df_bronze.columns:
    df_bronze = df_bronze.filter(F.year("pickup_datetime").isin(YEARS_TO_PROCESS))
records_in = df_bronze.count()

In [ ]:
valid_df, rejected_df = apply_rules(df_bronze, FHV_RULES)
records_valid = valid_df.count()
records_rejected = rejected_df.count()
print_rejection_summary(rejected_df)
control.log_quality_check("fhv_quality_rules", f"fhv_raw_years_{YEARS_TO_PROCESS}", records_in, records_rejected)

In [ ]:
if records_rejected > 0:
    audit_db = get_audit_db()
    rej_docs = []
    for row in rejected_df.collect():
        doc = row.asDict(recursive=True)
        doc["pipeline"] = "silver_fhv"
        doc["run_id"] = run_id
        rej_docs.append(doc)
    if rej_docs:
        audit_db["quarantine"].insert_many(rej_docs)

In [ ]:
zone_df = load_zone_lookup(spark)
valid_df = enrich_trip_locations(valid_df, zone_df)

In [ ]:
silver_df = build_fhv_silver(valid_df, run_id=run_id)
n_written = write_mongo(silver_df, settings.MONGO_DB_SILVER, "trips_fhv", mode="append")

In [ ]:
control.end(ExecutionStatus.SUCCESS, {"records_read": records_in, "records_written": n_written, "quarantined": records_rejected})

In [ ]:
import json
print(json.dumps(control.get_report(), indent=2, default=str))